# Ścieżka A · Dyskontowanie odroczonych nagród (5-trial)

**Projektowanie i badanie informatycznych systemów wspomagania decyzji** · notatnik projektu z aplikacją · szkielet do uzupełnienia

Notatnik prowadzi cały projekt ścieżki A w jednym miejscu, zgodnie z przewodnikiem *A — Dyskontowanie 5-trial — Przewodnik*:

| Etap | Kto robi | Pliki |
|---|---|---|
| 1–3 · eksport, pięć prób, ED50 i k | zespół (luki `# TODO`) | `qualtrics_raw.csv` → `responses_clean.csv` → `profil.csv` |
| 4–5 · model i nowe decyzje | zespół (luki `# TODO`) | `cases.csv` |
| 6 · zamrożenie, kontrakt, aplikacja | gotowe | `session_config.json`, `app.py` |
| 7 · analiza walidacji | zespół | wynik do raportu |

**Aplikacja jest gotowa.** Nie piszecie interfejsu. Aplikacja czyta pliki tworzone przez ten notatnik i działa tylko wtedy, gdy spełniają one kontrakt.

**Dwa tryby pracy** (komórka konfiguracji poniżej):
- `TRYB = 'przyklad'`: wymyślony uczestnik P01 z przewodnika. Testy sprawdzają liczby kontrolne z przewodnika. Zacznij tutaj.
- `TRYB = 'wlasne'`: eksport Waszego uczestnika. Testy funkcji działają dalej, a testy liczb P01 są pomijane: notatnik wypisuje wtedy Wasz wynik.

> **Jak pracować.** Uruchamiaj komórki po kolei. W miejscach `# TODO` uzupełnij kod i usuń `raise NotImplementedError`. Test pod każdym krokiem pokazuje ✓ albo zatrzymuje się na ✗ z opisem. Nie zmieniaj liczb w testach.

## 0 · Konfiguracja i folder projektu na Dysku Google
W Colabie pliki znikają po rozłączeniu, dlatego folder roboczy leży na Dysku Google. Przy pierwszym uruchomieniu Colab poprosi o zgodę na dostęp do Dysku. Folder można udostępnić zespołowi i prowadzącym. Dla przykładu i dla każdego uczestnika powstaje osobny folder.

W plikach są tylko kod uczestnika i jego odpowiedzi. Informacja przed zgodą powinna mówić, gdzie przechowujecie dane i kto ma do nich dostęp.

In [ ]:
TRYB = 'przyklad'          # 'przyklad' albo 'wlasne'
PARTICIPANT = 'P01'        # kod uczestnika (taki sam w Qualtricsie i w sesji)
SESSION_ID = 'S01'         # identyfikator sesji walidacyjnej
INSTRUCTOR_PIN = '2468'    # kod widoku prowadzącego w aplikacji; nie pokazuj go uczestnikowi
UZYJ_DYSKU = True          # w Colabie: folder projektu na Dysku Google

import os, sys, hashlib, inspect, json, math
from datetime import datetime, timezone
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

IN_COLAB = 'google.colab' in sys.modules
if TRYB not in ('przyklad', 'wlasne'):
    raise ValueError("TRYB musi mieć wartość 'przyklad' albo 'wlasne'")
if TRYB == 'przyklad':
    PARTICIPANT = 'P01'
if IN_COLAB and UZYJ_DYSKU:
    from google.colab import drive
    drive.mount('/content/drive')
    ROOT = '/content/drive/MyDrive/projekt_A'
else:
    ROOT = os.path.abspath('projekt_A')
WORKDIR = os.path.join(ROOT, 'przyklad_P01' if TRYB == 'przyklad' else PARTICIPANT)
os.makedirs(WORKDIR, exist_ok=True)
os.chdir(WORKDIR)
pd.set_option('display.width', 160); pd.set_option('display.max_columns', 40)

def check(label, cond):
    """Test ogólny: działa w obu trybach."""
    if not bool(cond):
        raise AssertionError('✗ ' + label)
    print('✓', label)

def check_example(label, cond, value=None):
    """Test liczb przykładu P01: w trybie 'wlasne' tylko wypisuje Wasz wynik."""
    if TRYB == 'przyklad':
        check(label, cond)
    else:
        print('ℹ pominięto test przykładu:', label, '' if value is None else f'| Wasz wynik: {value}')

def close(a, b, tol=1e-6):
    return a is not None and b is not None and abs(a - b) <= tol

def now_iso():
    return datetime.now(timezone.utc).isoformat(timespec='seconds')

def file_sha256(path):
    return hashlib.sha256(open(path, 'rb').read()).hexdigest()

def write_qualtrics_csv(path, rows, descriptions):
    """Plik w układzie eksportu Qualtricsa: 3 wiersze nagłówkowe + odpowiedzi."""
    cols = list(rows[0].keys())
    df = pd.DataFrame([[descriptions.get(c, c) for c in cols], [json.dumps({'ImportId': f'QID_{c}'}) for c in cols]]
                      + [[r.get(c, '') for c in cols] for r in rows], columns=cols)
    df.to_csv(path, index=False)

print(f'Tryb: {TRYB} | uczestnik: {PARTICIPANT} | folder: {WORKDIR}')

## 1 · Stałe procedury (rozdz. 4 i 6)
Kwoty są stałe: 500 zł teraz albo 1000 zł później. Drzewo ma 31 węzłów uporządkowanych według odroczenia i zaczyna od węzła 16 (3 tygodnie). Po LATER (kod 2) indeks rośnie, po NOW (kod 1) maleje, o 8, 4, 2 i 1. Piąta odpowiedź nie uruchamia szóstej próby. Wybiera jedną z dwóch wartości ED50 z tabeli 6 dla końcowego węzła.

In [ ]:
PROTOCOL = 'A_v1'
MODEL_VERSION = 'A_hyperbolic_v1'
NOW, LATER = 1, 2
A_LATER = 1000                      # zł, odroczona nagroda w profilowaniu i walidacji
K0 = 0.01                           # baseline: ta sama hiperbola, stałe k0 (1/dzień), ustalone przed testem
TIE_EPS = 0.01                      # |V − X| < 0,01 zł → TIE
K_P01 = 1 / 210.9                   # k przykładu P01: stała do testów funkcji
DELAY_LABELS = ['1 godzina', '2 godziny', '3 godziny', '4 godziny', '6 godzin', '9 godzin', '12 godzin', '1 dzień',
                '1,5 dnia', '2 dni', '3 dni', '4 dni', '1 tydzień', '1,5 tygodnia', '2 tygodnie', '3 tygodnie',
                '1 miesiąc', '2 miesiące', '3 miesiące', '4 miesiące', '6 miesięcy', '8 miesięcy', '1 rok', '2 lata',
                '3 lata', '4 lata', '5 lat', '8 lat', '12 lat', '18 lat', '25 lat']
# Tabela 6: końcowy węzeł → [ED50 po NOW, ED50 po LATER] w dniach
ED50_TABLE = {1: [0.04167, 0.05893], 3: [0.1021, 0.1444], 5: [0.2041, 0.3062], 7: [0.433, 0.7071],
              9: [1.225, 1.732], 11: [2.45, 3.464], 13: [5.292, 8.573], 15: [12.12, 17.15],
              17: [25.28, 43.05], 19: [74.56, 105.4], 21: [149.1, 210.9], 23: [298.2, 516.5],
              25: [894.7, 1265.0], 27: [1633, 2310], 29: [3579, 5368], 31: [7748, 9131]}
check('31 etykiet odroczeń i 16 węzłów końcowych', len(DELAY_LABELS) == 31 and len(ED50_TABLE) == 16)

## 2 · Eksport z Qualtricsa (rozdz. 8)
Drzewo ma 31 pól `A_N01`–`A_N31`, a jedna osoba odpowiada na pięć z nich. Puste pola oznaczają „nie wyświetlono” i są prawidłowe. W trybie `przyklad` P01 ma odpowiedzi 2, 1, 2, 1, 2 w węzłach 16, 24, 20, 22 i 21. Plik zawiera też wypełnienie testowe (`Status = 2`), które ma zostać odrzucone.

In [ ]:
def make_example_raw(path):
    answers_p01 = {16: 2, 24: 1, 20: 2, 22: 1, 21: 2}
    nodes = {f'A_N{i:02d}': answers_p01.get(i, '') for i in range(1, 32)}
    base = dict(ResponseId='R_P01', participant_id='P01', project_path='A', protocol_version=PROTOCOL, phase='profile',
                Status=0, Finished=1, Progress=100, **{'Duration (in seconds)': 233}, zgoda=1)
    rows = [{**base, 'ResponseId': 'R_test01', 'Status': 2, **{**nodes, 'A_N16': 1}}, {**base, **nodes}]
    write_qualtrics_csv(path, rows, {f'A_N{i:02d}': f'500 zł teraz czy 1000 zł za {DELAY_LABELS[i - 1]}?' for i in range(1, 32)})

In [ ]:
RAW_PATH = 'qualtrics_raw.csv'
if TRYB == 'przyklad':
    make_example_raw(RAW_PATH)
elif not os.path.exists(RAW_PATH):
    if not IN_COLAB:
        raise FileNotFoundError(f'Umieść eksport jako {os.path.join(WORKDIR, RAW_PATH)}')
    from google.colab import files
    up = files.upload()
    open(RAW_PATH, 'wb').write(next(iter(up.values())))
print('Plik źródłowy:', os.path.join(WORKDIR, RAW_PATH))

In [ ]:
def load_raw(path):
    """Eksport Qualtricsa: trzy wiersze nagłówkowe, dane od czwartego."""
    return pd.read_csv(path, skiprows=[1, 2])

def select_response(raw, participant_id, protocol_version, phase='profile'):
    """Dokładnie jedno wypełnienie: Status 0 (nie Preview 1, nie Test 2), Finished 1, zgoda 1, kod, wersja, faza."""
    # TODO: przefiltruj raw według sześciu warunków z opisu, sprawdź, że został dokładnie jeden wiersz, zwróć go
    raise NotImplementedError

raw = load_raw(RAW_PATH)
resp = select_response(raw, PARTICIPANT, PROTOCOL)
print('Wybrano wypełnienie', resp['ResponseId'])

In [ ]:
check_example('wybrano wypełnienie R_P01 (Test odrzucony)', resp['ResponseId'] == 'R_P01')

## 3 · Pięć prób, ED50 i k (rozdz. 6–8)
### 3.1 Od szerokiego rekordu do pięciu prób
Funkcja przechodzi drzewo od węzła 16 i za każdym razem czyta pole aktualnego węzła. Zwraca tabelę `trial, node_index, delay_label, choice_code` i status ścieżki:
- `complete`: pięć odpowiedzi w węzłach wynikających z poprzednich i żadne inne pole nie jest wypełnione;
- `incomplete`: brak odpowiedzi na któryś z pokazanych ekranów;
- `invalid`: wypełnione pole spoza ścieżki albo kod spoza {1, 2}.

In [ ]:
def trials_from_row(row):
    """Zwraca (DataFrame prób, status)."""
    # TODO: start w węźle 16, krok 8; pięć razy: odczytaj kod pola A_Nxx, zapisz wiersz,
    #       przesuń indeks (+krok po LATER, −krok po NOW) i podziel krok przez 2.
    #       Na końcu sprawdź, czy nie ma wypełnionych pól spoza ścieżki.
    raise NotImplementedError

def path_row(codes):
    """Szeroki rekord dla podanej ścieżki odpowiedzi (do testów). Gotowe."""
    r = {f'A_N{i:02d}': '' for i in range(1, 32)}; node, step = 16, 8
    for t, c in enumerate(codes):
        r[f'A_N{node:02d}'] = c
        if t < 4:
            node += step if c == LATER else -step; step //= 2
    return r

check('ścieżka 2,1,2,1,2 → węzły 16, 24, 20, 22, 21', list(trials_from_row(path_row([2, 1, 2, 1, 2]))[0].node_index) == [16, 24, 20, 22, 21])
_r = path_row([2, 1, 2, 1, 2])
check('brak piątej odpowiedzi → incomplete', trials_from_row({**_r, 'A_N21': ''})[1] == 'incomplete')
check('dodatkowy węzeł → invalid', trials_from_row({**_r, 'A_N08': 1})[1] == 'invalid')

trials, path_status = trials_from_row(resp)
check_example('ścieżka P01: 16 → 24 → 20 → 22 → 21, complete', (list(trials.node_index), path_status) == ([16, 24, 20, 22, 21], 'complete'),
              (list(trials.node_index) if len(trials) else [], path_status))
trials.insert(0, 'participant_id', PARTICIPANT)
trials.to_csv('responses_clean.csv', index=False)
trials

### 3.2 Scoring
ED50 to **wartość z tabeli** dla końcowego węzła i piątej odpowiedzi. Nie jest to ostatnio pokazane odroczenie ani średnia czasów. Dalej k = 1/ED50. Pięć odpowiedzi NOW daje dolny wynik graniczny, a pięć odpowiedzi LATER górny. Taki wynik oznacza ograniczenie rozdzielczości zadania, a nie punktową personalizację.

In [ ]:
def score(trials, status):
    """Zwraca dict(ed50_days, k_per_day, boundary_flag, profile_status)."""
    # TODO: dla statusu innego niż complete zwróć brak wyniku; w przeciwnym razie odczytaj ED50
    #       z ED50_TABLE[węzeł końcowy][0 po NOW / 1 po LATER], policz k i flagę granicy.
    raise NotImplementedError

_p = score(*trials_from_row(path_row([2, 1, 2, 1, 2])))
check('2,1,2,1,2 → ED50 = 210,9 dnia, k = 0,004741584', _p['ed50_days'] == 210.9 and close(_p['k_per_day'], 0.004741583689, 1e-12))
check('1,1,1,1,1 → dolny wynik graniczny (ED50 0,04167)', score(*trials_from_row(path_row([1] * 5)))['boundary_flag'] == 'lower')
check('2,2,2,2,2 → górny wynik graniczny (ED50 9131)', score(*trials_from_row(path_row([2] * 5)))['ed50_days'] == 9131)

prof = score(trials, path_status)
check_example('P01: ED50 = 210,9, profil valid', (prof['ed50_days'], prof['profile_status']) == (210.9, 'valid'), (prof['ed50_days'], prof['profile_status']))
profil = pd.DataFrame([dict(participant_id=PARTICIPANT, protocol_version=PROTOCOL,
                            final_node=int(trials.iloc[-1].node_index) if len(trials) else None,
                            last_choice=('LATER' if trials.iloc[-1].choice_code == LATER else 'NOW') if len(trials) else None, **prof,
                            amount_now=500, amount_later=A_LATER, delay_unit='day', k0=K0, model_version=MODEL_VERSION)])
profil.to_csv('profil.csv', index=False)
profil

## 4 · Model (rozdz. 10)
$V = \frac{A}{1 + kD}$ i porównanie z kwotą teraz X. `LATER` gdy V > X, `NOW` gdy V < X, `TIE` gdy |V − X| < 0,01 zł. Profil graniczny albo niepełny daje `NO_PREDICTION`. Jednostką D są **dni**, a funkcja odrzuca inne jednostki.

In [ ]:
def discounted_value(A, D, k):
    return A / (1 + k * D)

def predict_choice(X, D, k, A=A_LATER, unit='day', eps=TIE_EPS):
    """Zwraca dict(pred, V)."""
    # TODO: kontrola jednostki, brak k → NO_PREDICTION, V, porównanie z X i progiem eps
    raise NotImplementedError

def personal_k(profile):
    """k do personalizacji: tylko profil valid (wynik graniczny → None). Gotowe."""
    return profile['k_per_day'] if profile['profile_status'] == 'valid' else None

def explain_choice(X, D, k, A=A_LATER):
    """Wyjaśnienie pokazywane po serii. Gotowe."""
    if k is None:
        return 'Profil nie pozwala na personalizację.'
    V = discounted_value(A, D, k)
    rel = 'więcej' if V > X else 'mniej' if V < X else 'tyle samo'
    return (f'Według modelu z Twojego wcześniejszego badania {A} zł za {D} dni jest dla Ciebie warte około {V:.2f} zł teraz, '
            f'czyli {rel} niż {X} zł teraz. To przewidywanie preferencji w zadaniu, a nie wycena rynkowa.')

check('V(1000 zł, 90 dni) = 700,90 zł → LATER przy 600 zł', round(predict_choice(600, 90, K_P01)['V'], 2) == 700.90 and predict_choice(600, 90, K_P01)['pred'] == 'LATER')
check('przy 800 zł → NOW', predict_choice(800, 90, K_P01)['pred'] == 'NOW')
check('k0 = 0,01: V = 526,32 zł → NOW', round(predict_choice(600, 90, K0)['V'], 2) == 526.32 and predict_choice(600, 90, K0)['pred'] == 'NOW')
check('D = ED50 i X = A/2 → TIE', predict_choice(500, 210.9, K_P01)['pred'] == 'TIE')
check('V maleje, gdy D rośnie', all(np.diff([predict_choice(1, d, K_P01)['V'] for d in (1, 7, 30, 90, 365)]) < 0))
check('profil graniczny → NO_PREDICTION', predict_choice(600, 90, None)['pred'] == 'NO_PREDICTION')
try:
    predict_choice(600, 3, K_P01, unit='month'); check('zła jednostka odrzucona', False)
except ValueError:
    check('zła jednostka odrzucona', True)
k = personal_k(prof)
if k:
    D = np.linspace(0, 730, 400)
    plt.figure(figsize=(6, 3))
    plt.plot(D, discounted_value(A_LATER, D, k), label=f'k = {k:.6f} ({PARTICIPANT})')
    plt.plot(D, discounted_value(A_LATER, D, K0), '--', label='k0 = 0,01 (baseline)')
    plt.scatter([prof['ed50_days']], [500], zorder=3)
    plt.xlabel('odroczenie D [dni]'); plt.ylabel('V(1000 zł) [zł]'); plt.legend(); plt.title('Krzywa modelowa z jednego parametru, nie pomiar')
    plt.tight_layout(); plt.show()

## 5 · Nowe decyzje i `cases.csv` (rozdz. 11)
Bodźce pochodzą z przewodnika: 10 decyzji (przewodnik dopuszcza 8–12), zawsze 1000 zł później. LEFT oznacza kwotę teraz, RIGHT kwotę odroczoną. Przy k = 1/210,9 model przewiduje LATER w 5 próbach, baseline tylko w jednej, a modele różnią się w 4.

In [ ]:
STIMULI = [(7, 900), (7, 980), (30, 800), (30, 930), (90, 600), (90, 800), (180, 450), (180, 650), (365, 300), (365, 450)]

def build_cases(stimuli, k_personal, k0):
    """case_id, delay_days, amount_now, amount_later, V_personal, pred_personal, V_baseline, pred_baseline."""
    # TODO: jedna pętla po bodźcach, dwa wywołania predict_choice
    raise NotImplementedError

_c = build_cases(STIMULI, K_P01, K0)
check('k P01: LATER w 5 z 10 prób; baseline LATER tylko 7 dni / 900 zł; różnica w 4',
      (_c.pred_personal == 'LATER').sum() == 5 and list(_c.loc[_c.pred_baseline == 'LATER', 'case_id']) == ['A01']
      and (_c.pred_personal != _c.pred_baseline).sum() == 4)

cases = build_cases(STIMULI, k, K0)
rng = np.random.default_rng(20261020)
cases['display_order'] = rng.permutation(len(cases)) + 1
cases['left_text'] = cases.amount_now.map(lambda x: f'### {x} zł teraz')
cases['right_text'] = cases.delay_days.map(lambda d: f'### {A_LATER} zł za {d} dni')
cases['left_code'], cases['right_code'] = 'NOW', 'LATER'
cases['explanation'] = [explain_choice(r.amount_now, r.delay_days, k) for r in cases.itertuples()]
cases['model_version'] = MODEL_VERSION
cases['params'] = f'k={k:.12f}|k0={K0}|eps={TIE_EPS}' if k else f'k=None|k0={K0}'
PROFILE_OK = prof['profile_status'] == 'valid'
PROFILE_NOTE = prof['profile_status']
SESSION_EXTRA = dict(title='Teraz czy później?', left_button='Wybieram kwotę teraz', right_button='Wybieram 1000 zł później',
                     code_labels={'NOW': 'kwota teraz', 'LATER': '1000 zł później'},
                     intro='W każdej sytuacji wybierz: określona kwota teraz albo 1000 zł po podanym czasie. '
                           'Sytuacje są hipotetyczne: wyobraź sobie, że naprawdę otrzymasz wybraną kwotę. Nie ma poprawnych odpowiedzi.')
cases.sort_values('display_order')[['display_order', 'case_id', 'delay_days', 'amount_now', 'pred_personal', 'pred_baseline']]

## 6 · Zamrożenie, kontrakt i aplikacja

In [ ]:
# Opcjonalnie (rozszerzenie, np. własny interfejs): zapis przetestowanych funkcji do model.py.
# Aplikacja kursu nie potrzebuje tego pliku: opisy, predykcje i wyjaśnienia są w cases.csv.
EKSPORT_MODELU = False
if EKSPORT_MODELU:
    FUNCS = [trials_from_row, score, discounted_value, predict_choice, personal_k, explain_choice]
    CONSTS = ['NOW', 'LATER', 'A_LATER', 'K0', 'TIE_EPS', 'MODEL_VERSION', 'DELAY_LABELS', 'ED50_TABLE']
    src = ['"""model.py – wygenerowane z notatnika ścieżki A; nie edytuj ręcznie."""', 'import math', 'import pandas as pd', '']
    src += [f'{n} = {globals()[n]!r}' for n in CONSTS] + [''] + [inspect.getsource(f) for f in FUNCS]
    open('model.py', 'w', encoding='utf-8').write('\n'.join(src))
    sys.path.insert(0, WORKDIR)
    if 'model' in sys.modules: del sys.modules['model']
    import model
    check('model.py: V(1000, 90) = 700,90', round(model.predict_choice(600, 90, K_P01)['V'], 2) == 700.90)

## Zamrożenie przed sesją
Komórka zapisuje `cases.csv` z opisami opcji i predykcjami obu modeli oraz `session_config.json` ze skrótem SHA-256. Skrót wpisz do dziennika **przed sesją**. W trybie `wlasne` notatnik nie nadpisze zamrożonych przypadków: jeśli po zamrożeniu zmienisz dane albo kod, komórka zatrzyma się z błędem. 

In [ ]:
new_csv = cases.to_csv(index=False).encode()
new_hash = hashlib.sha256(new_csv).hexdigest()
if not PROFILE_OK:
    raise RuntimeError(f'Profil nie pozwala na personalizację ({PROFILE_NOTE}). Nie zamrażamy przypadków i nie prowadzimy sesji.')
if TRYB == 'wlasne' and os.path.exists('session_config.json'):
    frozen = json.load(open('session_config.json', encoding='utf-8'))
    if frozen['frozen_sha256'] != new_hash:
        raise RuntimeError('Przypadki są już zamrożone, a nowe obliczenia dają inny cases.csv. Nie nadpisuję. '
                           'Jeśli zmiana nastąpiła PRZED sesją, usuń świadomie session_config.json i opisz to w dzienniku; '
                           'po sesji wymaga to nowej wersji modelu i nowego zestawu.')
    print('Zamrożenie bez zmian:', new_hash)
else:
    open('cases.csv', 'wb').write(new_csv)
    json.dump(dict(contract='A_v1', participant_id=PARTICIPANT, session_id=SESSION_ID, frozen_sha256=new_hash,
                   frozen_at=now_iso(), model_version=MODEL_VERSION, seed=20261020, instructor_pin=INSTRUCTOR_PIN,
                   profile_ok=PROFILE_OK, profile_note=PROFILE_NOTE, **SESSION_EXTRA),
              open('session_config.json', 'w', encoding='utf-8'), ensure_ascii=False, indent=1)
    print('Zamrożono cases.csv | SHA-256:', new_hash)
FROZEN_HASH = file_sha256('cases.csv')

In [ ]:
cfg = json.load(open('session_config.json', encoding='utf-8'))
_cases = pd.read_csv('cases.csv')
missing = [c for c in ['case_id', 'display_order', 'left_text', 'right_text', 'left_code', 'right_code', 'pred_personal', 'pred_baseline', 'model_version', 'params', 'explanation'] if c not in _cases.columns]
check('cases.csv: wymagane kolumny', not missing or print('Brakuje:', missing))
check('session_config.json: kontrakt i zgodny skrót cases.csv', cfg['contract'] == 'A_v1' and cfg['frozen_sha256'] == file_sha256('cases.csv'))
check('profil.csv istnieje', os.path.exists('profil.csv'))

## Aplikacja i sesja z uczestnikiem
Komórka z kodem aplikacji jest zwinięta. **Nie edytuj jej.** Zapisuje plik `app.py` w folderze projektu.

**Przebieg sesji:**
1. Uczestnik widzi kolejne pary „kwota teraz / 1000 zł później” i wybiera bez podpowiedzi. Każdy wybór trafia od razu do `validation.csv`.
2. Po wszystkich wyborach aplikacja pokazuje rekomendacje i wyjaśnienia, a potem zbiera jedną ocenę dopasowania i jedną ocenę użyteczności serii (1–7).

Widok prowadzącego (panel boczny, kod `INSTRUCTOR_PIN`) pokazuje kontrakt, zamrożone przypadki i log. Nie otwieraj go przy uczestniku. Odświeżenie strony nie gubi danych: aplikacja wznawia sesję od pierwszego niezapisanego kroku.

In [ ]:
#@title Kod aplikacji – gotowy, nie edytuj { display-mode: "form" }
APP_SOURCE = r'''"""Sesja walidacyjna – ścieżki A, B, C i E (wybór jednej z dwóch opcji).

Gotowa aplikacja kursu: zespół jej NIE edytuje. Aplikacja niczego nie przewiduje sama:
opisy opcji, predykcje i wyjaśnienia pochodzą z zamrożonego cases.csv utworzonego w notatniku.

Uruchomienie:  streamlit run app.py -- --data <folder z plikami projektu>
Folder musi zawierać: cases.csv, profil.csv i session_config.json (tworzy je notatnik).
Aplikacja zapisuje: validation.csv (po każdym wyborze) i session_<id>.json (oceny serii).
"""
import argparse
import hashlib
import json
import os
import sys
from datetime import datetime, timezone

import pandas as pd
import streamlit as st

APP_VERSION = 'pred_app_v1'
REQUIRED_CASES = ['case_id', 'display_order', 'left_text', 'right_text', 'left_code', 'right_code',
                  'pred_personal', 'pred_baseline', 'model_version', 'params', 'explanation']
REQUIRED_CFG = ['contract', 'participant_id', 'session_id', 'frozen_sha256', 'instructor_pin', 'profile_ok',
                'title', 'intro', 'left_button', 'right_button', 'code_labels']


def data_dir():
    p = argparse.ArgumentParser()
    p.add_argument('--data', default='.')
    return os.path.abspath(p.parse_known_args(sys.argv[1:])[0].data)


DATA = data_dir()
path = lambda name: os.path.join(DATA, name)
now_iso = lambda: datetime.now(timezone.utc).isoformat(timespec='seconds')
sha256 = lambda p: hashlib.sha256(open(p, 'rb').read()).hexdigest()


def load_project():
    """Wczytuje pliki i sprawdza kontrakt. Zwraca (cfg, cases, błędy, ostrzeżenia)."""
    errors, warnings = [], []
    for name in ['cases.csv', 'profil.csv', 'session_config.json']:
        if not os.path.exists(path(name)):
            errors.append(f'Brak pliku {name} w folderze {DATA}. Uruchom notatnik do sekcji „Zamrożenie”.')
    if errors:
        return {}, None, errors, warnings
    cfg = json.load(open(path('session_config.json'), encoding='utf-8'))
    missing = [k for k in REQUIRED_CFG if k not in cfg]
    if missing:
        errors.append(f'session_config.json: brak pól {", ".join(missing)}')
    cases = pd.read_csv(path('cases.csv'), dtype={'case_id': str})
    missing = [c for c in REQUIRED_CASES if c not in cases.columns]
    if missing:
        errors.append(f'cases.csv: brak kolumn {", ".join(missing)}')
    if sha256(path('cases.csv')) != cfg.get('frozen_sha256'):
        errors.append('cases.csv różni się od wersji zamrożonej (SHA-256 nie zgadza się z session_config.json).')
    if not cfg.get('profile_ok', False):
        errors.append(f'Profil nie pozwala na personalizację: {cfg.get("profile_note", "")}')
    if not errors:
        bad = set(cases.pred_personal.astype(str)) - set(cfg['code_labels']) - {'TIE', 'NO_PREDICTION'}
        if bad:
            errors.append(f'cases.csv: nieznane kody predykcji {bad}')
        cases = cases.sort_values('display_order').reset_index(drop=True)
    return cfg, cases, errors, warnings


def read_log():
    if os.path.exists(path('validation.csv')):
        return pd.read_csv(path('validation.csv'), dtype={'session_id': str, 'case_id': str})
    return pd.DataFrame(columns=['session_id', 'case_id'])


def append_log(row):
    log = pd.concat([read_log(), pd.DataFrame([row])], ignore_index=True)
    tmp = path('validation.csv.tmp')
    log.to_csv(tmp, index=False)
    os.replace(tmp, path('validation.csv'))       # zapis atomowy


st.set_page_config(page_title='Sesja walidacyjna', page_icon='🧭', layout='wide')
cfg, cases, errors, warnings = load_project()
SESSION = str(cfg.get('session_id', 'S01'))
SESSION_JSON = path(f'session_{SESSION}.json')

with st.sidebar:
    st.caption(f'{APP_VERSION} · kontrakt {cfg.get("contract", "?")}')
    pin = st.text_input('Kod prowadzącego', type='password')
    instructor = bool(cfg) and pin != '' and pin == str(cfg.get('instructor_pin'))

if instructor:
    st.title('Widok prowadzącego')
    for e in errors:
        st.error(e)
    for w in warnings:
        st.warning(w)
    if not errors:
        st.success(f'Kontrakt spełniony · cases.csv zgodny z zamrożeniem · SHA-256 {cfg["frozen_sha256"][:16]}…')
        st.subheader('Profil')
        st.dataframe(pd.read_csv(path('profil.csv')), hide_index=True)
        st.subheader('Zamrożone przypadki')
        st.dataframe(cases[['display_order', 'case_id', 'left_text', 'right_text', 'pred_personal', 'pred_baseline']], hide_index=True)
    log = read_log()
    n = int((log.session_id.astype(str) == SESSION).sum()) if len(log) else 0
    st.subheader(f'Log sesji {SESSION}: {n} zapisanych wyborów')
    st.dataframe(log, hide_index=True)
    if os.path.exists(path('validation.csv')):
        st.download_button('Pobierz validation.csv', open(path('validation.csv'), 'rb').read(), 'validation.csv', 'text/csv')
    if os.path.exists(SESSION_JSON):
        st.json(json.load(open(SESSION_JSON, encoding='utf-8')))
    st.stop()

st.title(cfg.get('title', 'Sesja walidacyjna'))
if errors:
    st.error('Aplikacja nie jest gotowa do sesji. Poproś prowadzącego o sprawdzenie (widok prowadzącego w panelu bocznym).')
    st.stop()

log = read_log()
done = log[log.session_id.astype(str) == SESSION]
remaining = cases[~cases.case_id.isin(set(done.case_id.astype(str)))]
if os.path.exists(SESSION_JSON):
    st.success('Sesja zakończona. Dziękujemy za udział!')
    st.stop()

if 'started' not in st.session_state:
    st.session_state.started = len(done) > 0
if not st.session_state.started:
    st.write(cfg['intro'])
    st.write(f'Zobaczysz {len(cases)} sytuacji. W każdej wybierz jedną z dwóch możliwości. '
             'Wyjaśnienia działania systemu zobaczysz dopiero po zapisaniu wszystkich wyborów.')
    if st.button('Rozpocznij', type='primary'):
        st.session_state.started = True
        st.session_state.started_at = now_iso()
        st.rerun()
    st.stop()

if len(remaining):
    row = remaining.iloc[0]
    k = len(done) + 1
    st.progress((k - 1) / len(cases), text=f'Sytuacja {k} z {len(cases)}')
    st.caption(cfg['intro'])
    cl, cr = st.columns(2)
    for col, txt in ((cl, row.left_text), (cr, row.right_text)):
        with col:
            with st.container(border=True):
                st.markdown(txt)

    def choose(position):
        code = row.left_code if position == 'left' else row.right_code
        rec = dict(participant_id=cfg['participant_id'], session_id=SESSION, case_id=str(row.case_id),
                   display_order=int(row.display_order), chosen_position=position, answer=code, answered_at=now_iso(),
                   model_version=row.model_version, params=row.params, app_version=APP_VERSION)
        rec.update({c: row[c] for c in cases.columns if c.startswith('pred_')})
        append_log(rec)

    b1, b2 = st.columns(2)
    b1.button(cfg['left_button'], key=f'L_{row.case_id}', on_click=choose, args=('left',), use_container_width=True)
    b2.button(cfg['right_button'], key=f'R_{row.case_id}', on_click=choose, args=('right',), use_container_width=True)
    st.stop()

st.progress(1.0, text='Wszystkie wybory zapisane')
st.header('Jak system opisuje Twoje wybory')
st.write('Teraz, po zapisaniu wszystkich decyzji, pokazujemy rekomendacje systemu i ich uzasadnienie. '
         'Rekomendacja opisuje działanie modelu, a nie poprawność Twojej decyzji.')
ans = dict(zip(done.case_id.astype(str), done.answer))
labels = cfg['code_labels']
for _, row in cases.iterrows():
    mine = ans.get(str(row.case_id))
    with st.expander(f'Sytuacja {int(row.display_order)}: wybrano „{labels.get(mine, mine)}”'):
        c1, c2 = st.columns(2)
        c1.markdown(row.left_text); c2.markdown(row.right_text)
        rec = row.pred_personal
        if rec in labels:
            st.write(f'**System rekomendowałby: {labels[rec]}.** {row.explanation}')
        else:
            st.write(f'System nie rozstrzyga tej sytuacji ({rec}). {row.explanation}')

st.header('Ocena całej serii')
fit = st.slider('Na ile rekomendacje były dopasowane do Twoich wyborów? (1 = wcale, 7 = bardzo)', 1, 7, 4)
useful = st.slider('Na ile wyjaśnienia były dla Ciebie użyteczne? (1 = wcale, 7 = bardzo)', 1, 7, 4)
if st.button('Zakończ sesję', type='primary'):
    json.dump(dict(participant_id=cfg['participant_id'], session_id=SESSION, rating_fit=int(fit), rating_usefulness=int(useful),
                   started_at=st.session_state.get('started_at', ''), finished_at=now_iso(), n_choices=len(done),
                   cases_sha256=cfg['frozen_sha256'], app_version=APP_VERSION),
              open(SESSION_JSON, 'w', encoding='utf-8'), ensure_ascii=False, indent=1)
    st.rerun()
'''
open('app.py', 'w', encoding='utf-8').write(APP_SOURCE)
print('Zapisano app.py w', WORKDIR)

In [ ]:
#@title ▶ Uruchom aplikację w Colabie { display-mode: "form" }
URUCHOM = False  #@param {type:"boolean"}
import subprocess, socket, time
def _port_open(port):
    with socket.socket() as s:
        return s.connect_ex(('127.0.0.1', port)) == 0
if URUCHOM:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'streamlit'], check=True)
    if not _port_open(8501):
        subprocess.Popen([sys.executable, '-m', 'streamlit', 'run', os.path.join(WORKDIR, 'app.py'),
                          '--server.port', '8501', '--server.headless', 'true',
                          '--server.enableCORS', 'false', '--server.enableXsrfProtection', 'false',
                          '--', '--data', WORKDIR],
                         stdout=open(os.path.join(WORKDIR, 'streamlit.log'), 'w'), stderr=subprocess.STDOUT)
        for _ in range(60):
            if _port_open(8501): break
            time.sleep(1)
    if IN_COLAB:
        from google.colab import output
        output.serve_kernel_port_as_window(8501, anchor_text='▶ Otwórz aplikację w nowej karcie')
    else:
        print('Aplikacja: http://localhost:8501')
else:
    print('Zaznacz URUCHOM i uruchom komórkę ponownie. Jeśli link nie działa, przeprowadź sesję lokalnie (komórka niżej).')

**Sesja lokalna (zalecana przy uczestniku).** Colab może się rozłączyć w trakcie sesji. Pewniej jest uruchomić aplikację na laptopie:
1. Uruchom komórkę poniżej i pobierz `pakiet_sesji_A.zip` (`app.py`, `profil.csv`, `cases.csv`, `session_config.json`).
2. Na laptopie: `pip install streamlit pandas`, potem w rozpakowanym folderze `streamlit run app.py -- --data .`
3. Po sesji skopiuj `validation.csv` i `session_S01.json` do folderu projektu na Dysku i uruchom analizę w trybie `wlasne`.

In [ ]:
import zipfile
with zipfile.ZipFile('pakiet_sesji_A.zip', 'w') as z:
    for f in ['app.py', 'profil.csv', 'cases.csv', 'session_config.json']:
        z.write(f)
print('Zapisano', os.path.join(WORKDIR, 'pakiet_sesji_A.zip'))
if IN_COLAB:
    from google.colab import files
    files.download('pakiet_sesji_A.zip')

## 7 · Analiza walidacji (rozdz. 11)
- `przyklad`: notatnik tworzy log w formacie aplikacji z **syntetycznymi** odpowiedziami.
- `wlasne`: notatnik czyta `validation.csv` i `session_S01.json` zapisane przez aplikację.

In [ ]:
UNAMBIGUOUS = lambda s: s.notna() & ~s.isin(['TIE', 'NO_PREDICTION'])

def summarize(df, pred='pred_personal', base='pred_baseline', ans='answer'):
    """Podsumowanie walidacji wg przewodnika.

    - pokrycie: liczba prób z jednoznaczną predykcją (bez TIE i NO_PREDICTION),
    - trafność obu modeli na WSPÓLNYM zbiorze prób z odpowiedzią i jednoznaczną predykcją obu,
    - podział 2×2: oba trafiły / tylko indywidualny / tylko bazowy / żaden,
    - liczba prób wspólnego zbioru, w których modele się różnią.
    """
    # TODO: 1) wybierz wspólny zbiór: odpowiedź jest, obie predykcje jednoznaczne (UNAMBIGUOUS),
    #       2) policz trafienia obu modeli na tym zbiorze i podział 2×2,
    #       3) zwróć słownik o kluczach: n_trials, n_answered, coverage_personal,
    #          coverage_baseline, n_common, hits_personal, hits_baseline, both,
    #          only_personal, only_baseline, none, n_differ.
    raise NotImplementedError

def report(s, name_p='model indywidualny', name_b='model bazowy'):
    pct = lambda k, m: f'{k}/{m} = {100*k/m:.0f}%' if m else f'{k}/{m}'
    print(f"Pokrycie: {name_p} {s['coverage_personal']}/{s['n_trials']}, {name_b} {s['coverage_baseline']}/{s['n_trials']}")
    print(f"Trafność na wspólnym zbiorze: {name_p} {pct(s['hits_personal'], s['n_common'])}, "
          f"{name_b} {pct(s['hits_baseline'], s['n_common'])}")
    print(f"2×2: oba {s['both']}, tylko indywidualny {s['only_personal']}, tylko bazowy {s['only_baseline']}, żaden {s['none']}")
    print(f"Próby wspólnego zbioru, w których modele się różnią: {s['n_differ']}/{s['n_common']}")
    print('TIE i NO_PREDICTION nie są ani trafieniem, ani pomyłką. Wynik opisuje test jednej osoby.')

In [ ]:
_t = pd.DataFrame({'pred_personal': ['A', 'B', 'TIE', 'A', 'A'],
                   'pred_baseline': ['A', 'A', 'B', 'NO_PREDICTION', 'B'],
                   'answer':        ['A', 'B', 'B', 'A', None]})
_s = summarize(_t)
check('pokrycie liczy tylko jednoznaczne predykcje', (_s['coverage_personal'], _s['coverage_baseline']) == (4, 4))
check('wspólny zbiór pomija TIE, NO_PREDICTION i brak odpowiedzi', _s['n_common'] == 2)
check('podział 2×2', (_s['both'], _s['only_personal'], _s['only_baseline'], _s['none']) == (1, 1, 0, 0))

In [ ]:
def synthetic_session():
    ans = dict(zip([f'A{i:02d}' for i in range(1, 11)], ['LATER', 'NOW', 'LATER', 'NOW', 'LATER', 'NOW', 'NOW', 'NOW', 'LATER', 'NOW']))
    v = pd.read_csv('cases.csv', dtype={'case_id': str}).sort_values('display_order')
    log = v[['case_id', 'display_order', 'model_version', 'params', 'pred_personal', 'pred_baseline']].copy()
    log.insert(0, 'participant_id', PARTICIPANT); log.insert(1, 'session_id', SESSION_ID)
    log['answer'] = log.case_id.map(ans)
    log['chosen_position'] = np.where(log.answer == 'NOW', 'left', 'right')
    log['answered_at'] = now_iso(); log['app_version'] = 'synthetic'
    return log, dict(rating_fit=6, rating_usefulness=5, n_choices=len(log))

In [ ]:
check('cases.csv nie zmienił się od zamrożenia', file_sha256('cases.csv') == FROZEN_HASH)
validation, session = None, None
if TRYB == 'przyklad':
    validation, session = synthetic_session()          # odpowiedzi syntetyczne w formacie logu aplikacji
elif os.path.exists('validation.csv'):
    validation = pd.read_csv('validation.csv', dtype={'session_id': str, 'case_id': str})
    validation = validation[validation.session_id == SESSION_ID]
    if os.path.exists(f'session_{SESSION_ID}.json'):
        session = json.load(open(f'session_{SESSION_ID}.json', encoding='utf-8'))
    _frozen = pd.read_csv('cases.csv', dtype={'case_id': str})
    check('log zawiera wyłącznie zamrożone przypadki', set(validation.case_id) <= set(_frozen.case_id.astype(str)))
    if 'pred_personal' in validation:
        _m = validation.merge(_frozen, on='case_id', suffixes=('', '_f'))
        check('predykcje w logu zgodne z zamrożonymi', ((_m.pred_personal == _m.pred_personal_f) & (_m.pred_baseline == _m.pred_baseline_f)).all())
    print(f'Wczytano {len(validation)} zapisów z sesji {SESSION_ID}.', '' if session else '⚠ Brak pliku session_*.json.')
else:
    print('Brak validation.csv: najpierw przeprowadź sesję.')

In [ ]:
if validation is not None:
    s = summarize(validation)
    report(s, 'hiperbola z k osoby', 'hiperbola z k0 = 0,01')
    check_example('trafność 9/10 wobec 7/10, 2×2 = 6/3/1/0', (s['hits_personal'], s['hits_baseline'], s['n_common'], s['both'],
                  s['only_personal'], s['only_baseline'], s['none']) == (9, 7, 10, 6, 3, 1, 0))

In [ ]:
if session:
    print('Ocena dopasowania serii (1–7):', session.get('rating_fit'), '| ocena użyteczności serii (1–7):', session.get('rating_usefulness'))

## 8 · Rozszerzenie: zależność od k0 i od rozdzielczości tabeli
Analizę dodatkową trzeba zapisać **przed sesją**. Sąsiednia wartość tabeli dla węzła końcowego pokazuje, które predykcje zależą od jednej odpowiedzi.

In [ ]:
# TODO (opcjonalne): predykcje dla k0 ∈ {0,005; 0,01; 0,02} i dla sąsiedniej wartości ED50 z tabeli

## 9 · Pytania do raportu
1. Dlaczego zapisanie ostatniego pokazanego odroczenia (6 miesięcy) jako ED50 jest błędem?
2. Co oznacza, że baseline k0 = 0,01 przewiduje prawie zawsze „teraz”? Jak to wpływa na interpretację przewagi modelu?
3. Dlaczego wynik graniczny prowadzi do odmowy personalizacji, a nie do użycia skrajnej wartości tabeli?